# Models: baselines and the course model

This notebook fits each model on the training split and scores it on the validation split. Every model uses the same interface and the same two metrics, so the rows of the results table are comparable.

The trivial baseline gives every row the class frequencies of the training set, so it always answers "fit". It sets the floor: a model that cannot beat it has learned nothing.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd  # noqa: E402

from src import data, metrics, models  # noqa: E402

## Data and split

`src.data.load()` reads the raw file, or the small sample when `DATA_DIR` points at `tests/fixtures`. `src.data.split()` returns the shared 80/10/10 split with seed 158. This notebook leaves the test split alone; the final evaluation scores it once.

In [ ]:
df = data.load()
train, valid, test = data.split(df)
print(f"train {len(train)}, valid {len(valid)}")
train["fit"].value_counts(normalize=True).round(3)

In [ ]:
train_p, valid_p, _ = data.preprocess(
    train, valid, test
)  # fills use train statistics only; test is not scored here

## Fit and score

Every model returns a probability for each class, because the primary metric (average one-vs-rest AUC) ranks scores. Macro-F1 uses the hard labels, which are the most likely class. The majority baseline gives the same probabilities to every row, so its AUC is 0.5 by construction. Macro-F1 shows what always saying "fit" costs: the two minority classes score 0.

In [ ]:
baseline = models.MajorityClass().fit(train_p)
baseline_row = {
    "average AUC": metrics.average_auc(valid_p["fit"], baseline.predict_proba(valid_p)),
    "macro-F1": metrics.macro_f1(valid_p["fit"], baseline.predict(valid_p)),
}
baseline_row

## Course model: multinomial logistic regression

**Inputs.** One row per transaction after `data.preprocess()`: the garment `size`, the user's `weight_lbs`, `height_in`, `bust_band`, `cup` and `age` with a 0/1 missing flag for each, and the one-hot columns `body_type`, `category` and `rented_for`. Numbers are standardised. The rating and the review text are not used, because they are written after the fit is known.

**Output.** A probability for each of small, fit and large, taken from a softmax over three linear scores $z_k = w_k^\top x + b_k$:

$$p(y = k \mid x) = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

**Objective.** Fit $W$ and $b$ on the training split by minimising the L2-regularised cross-entropy:

$$\min_{W,b}\; \frac{1}{2C}\lVert W \rVert_2^2 + \sum_{i=1}^{n} -\log p(y_i \mid x_i)$$

$C = 1$ and equal class weights are the defaults. They are not tuned here. The model sees a user's measurements and the size, but no user or item identity, so it cannot learn that a particular dress runs small.

In [ ]:
logreg = models.LogisticRegression().fit(train_p)
logreg_row = {
    "average AUC": metrics.average_auc(valid_p["fit"], logreg.predict_proba(valid_p)),
    "macro-F1": metrics.macro_f1(valid_p["fit"], logreg.predict(valid_p)),
}
results = pd.DataFrame([baseline_row, logreg_row], index=["MajorityClass", "LogisticRegression"]).round(4)
print(results)
results

## Reading the table

Compare the two rows on both metrics. The majority baseline scores an AUC of exactly 0.5, so any AUC above 0.5 for logistic regression is signal from size and measurements. Macro-F1 can still stay low if the model rarely predicts the rare classes: with about 74% of labels being "fit", the most likely class is usually "fit". The numbers in this table come from whichever file `DATA_DIR` points at, so quote them only after a run on the full data.

## Adding a model

A new model subclasses `models.Model`, implements `fit(train)` and `predict_proba(df)`, and gets `predict` for free. Add a row to the results table and nothing else changes.